Looks good. Can move on to setting up the graph data. Because this data isn't as clean as the original, we're going to have to edit the original functions to add a try except block to catch the errors.

The script below calculates the graph data in a distributed way, but it needs the compounds to all have material ids, which they currently do not. 

In [1]:
import pandas as pd
import numpy as np 

In [2]:

data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'

#load in the data 
train_df = pd.read_csv(data_dir + 'train_xrd.csv')
test_df = pd.read_csv(data_dir + 'test_xrd.csv')
val_df = pd.read_csv(data_dir + 'val_xrd.csv')

In [3]:
# Manually assign names
train_df.name = 'train'
test_df.name = 'test'
val_df.name = 'val'

# Function to add material_id column
def add_material_id(df):
    df['material_id'] = df.apply(lambda row: f"{df.name}_not_true_mat_id_{row.name}", axis=1)

In [5]:
# Apply the function to each DataFrame
add_material_id(train_df)
add_material_id(test_df)
add_material_id(val_df)

In [7]:
#save the dataframes to csv
train_df.to_csv(data_dir + 'train_xrd.csv', index=False)
test_df.to_csv(data_dir + 'test_xrd.csv', index=False)
val_df.to_csv(data_dir + 'val_xrd.csv', index=False)

Before you send out to workers, test the code as shown below

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import torch
import copy
import itertools

from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.graphs import StructureGraph
from pymatgen.analysis import local_env

from networkx.algorithms.components import is_connected

from sklearn.metrics import accuracy_score, recall_score, precision_score

from torch_scatter import scatter

from p_tqdm import p_umap

import ast
#import the random function library
import random

import os 

from tqdm.auto import tqdm
tqdm.pandas()

CrystalNN = local_env.CrystalNN(
    distance_cutoffs=None, x_diff_weight=-1, porous_adjustment=False)

from cdvae.common.data_utils import * 

import sys

#read in the worker number 
try: 
    worker_num = int(sys.argv[1])
except: 
    worker_num = 0
data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'

#load in the data 
train_df = pd.read_csv(data_dir + 'train_xrd.csv')
test_df = pd.read_csv(data_dir + 'test_xrd.csv')
val_df = pd.read_csv(data_dir + 'val_xrd.csv')
def build_crystal(crystal_str, niggli=True, primitive=False):
    try: 
        """Build crystal from cif string."""
        crystal = Structure.from_str(crystal_str, fmt='cif')

        if primitive:
            crystal = crystal.get_primitive_structure()

        if niggli:
            crystal = crystal.get_reduced_structure()

        canonical_crystal = Structure(
            lattice=Lattice.from_parameters(*crystal.lattice.parameters),
            species=crystal.species,
            coords=crystal.frac_coords,
            coords_are_cartesian=False,
        )
        # match is gaurantteed because cif only uses lattice params & frac_coords
        # assert canonical_crystal.matches(crystal)
        return canonical_crystal
    except: 
        return None 


def build_crystal_graph(crystal, graph_method='crystalnn'):
    try: 
        """
        """

        if graph_method == 'crystalnn':
            crystal_graph = StructureGraph.with_local_env_strategy(
                crystal, CrystalNN)
        elif graph_method == 'none':
            pass
        else:
            raise NotImplementedError

        frac_coords = crystal.frac_coords
        atom_types = crystal.atomic_numbers
        lattice_parameters = crystal.lattice.parameters
        lengths = lattice_parameters[:3]
        angles = lattice_parameters[3:]

        assert np.allclose(crystal.lattice.matrix,
                        lattice_params_to_matrix(*lengths, *angles))

        edge_indices, to_jimages = [], []
        if graph_method != 'none':
            for i, j, to_jimage in crystal_graph.graph.edges(data='to_jimage'):
                edge_indices.append([j, i])
                to_jimages.append(to_jimage)
                edge_indices.append([i, j])
                to_jimages.append(tuple(-tj for tj in to_jimage))

        atom_types = np.array(atom_types)
        lengths, angles = np.array(lengths), np.array(angles)
        edge_indices = np.array(edge_indices)
        to_jimages = np.array(to_jimages)
        num_atoms = atom_types.shape[0]

        return frac_coords, atom_types, lengths, angles, edge_indices, to_jimages, num_atoms
    except: 
        return None
data_frames = {"train": train_df, "test": test_df, "val": val_df}

for name, df in data_frames.items():

    #assuming you want 20 workers 
    num_splits = 100 # 1000 just for testing 
    chunk_size = np.ceil(len(df)/num_splits)
    
    start_index = int(worker_num*chunk_size)
    end_index = int(min(start_index + chunk_size, len(df))) #prevents end index > len(df)
    sub_df = df.iloc[start_index:end_index].copy()
    sub_crystals = sub_df['cif'].progress_apply(build_crystal)
    sub_graphs = sub_crystals.progress_apply(build_crystal_graph)

    materials_ids = sub_df['material_id'].values

    #make a dictionary using the materials_ids as keys and the graphs as values
    graph_dict = dict(zip(materials_ids, sub_graphs))

    #save the dictionary to a file
    torch.save(graph_dict, data_dir + 'graph_preloading_data/{}_{}.pt'.format(name, worker_num))
    
    print('Saved {}_{}.pt'.format(name, worker_num))

Read the pickle files back in to make sure everything worked ok

In [ ]:
import numpy as np
import torch 
from tqdm import tqdm
import pandas as pd
#assuming all of the data was stored as .pt dictionaries of string keys and graph values
num_workers = 20 
dataset_names =  ['train', 'test', 'val']
data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'


In [ ]:
worker_num = 0
name = "train"
train_chunk = torch.load(data_dir + 'graph_preloading_data/{}_{}.pt'.format(name, worker_num))

In [ ]:
len(train_chunk)

2264

In [ ]:
worker_num = 0
name = "test"
test_chunk = torch.load(data_dir + 'graph_preloading_data/{}_{}.pt'.format(name, worker_num))

In [ ]:
len(test_chunk)

755

In [ ]:
worker_num = 0
name = "val"
val_chunk = torch.load(data_dir + 'graph_preloading_data/{}_{}.pt'.format(name, worker_num))

In [ ]:
len(val_chunk)

755

In order to double check that the graphs were created correctly and loaded correctly, we should calculate a few graphs locally and compare 

In [ ]:
def compare_complex_structures(struct1, struct2):
    # Check if structures have the same length
    if len(struct1) != len(struct2):
        return False

    for elem1, elem2 in zip(struct1, struct2):
        # Check if both elements are NumPy arrays
        if isinstance(elem1, np.ndarray) and isinstance(elem2, np.ndarray):
            if not np.array_equal(elem1, elem2):
                return False
        # Check if elements are the same (for non-array types)
        elif elem1 != elem2:
            return False

    return True


In [9]:
data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'

#load in the data 
train_df = pd.read_csv(data_dir + 'train_xrd.csv')
test_df = pd.read_csv(data_dir + 'test_xrd.csv')
val_df = pd.read_csv(data_dir + 'val_xrd.csv')

In [10]:
from tqdm import tqdm

In [12]:
import numpy as np
import pandas as pd
import networkx as nx
import torch
import copy
import itertools

from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.graphs import StructureGraph
from pymatgen.analysis import local_env

from networkx.algorithms.components import is_connected

from sklearn.metrics import accuracy_score, recall_score, precision_score

from torch_scatter import scatter

from p_tqdm import p_umap

import ast
#import the random function library
import random

import os 

from tqdm.auto import tqdm
tqdm.pandas()

CrystalNN = local_env.CrystalNN(
    distance_cutoffs=None, x_diff_weight=-1, porous_adjustment=False)

from cdvae.common.data_utils import * 

import sys


def build_crystal(crystal_str, niggli=True, primitive=False):
    try: 
        """Build crystal from cif string."""
        crystal = Structure.from_str(crystal_str, fmt='cif')

        if primitive:
            crystal = crystal.get_primitive_structure()

        if niggli:
            crystal = crystal.get_reduced_structure()

        canonical_crystal = Structure(
            lattice=Lattice.from_parameters(*crystal.lattice.parameters),
            species=crystal.species,
            coords=crystal.frac_coords,
            coords_are_cartesian=False,
        )
        # match is gaurantteed because cif only uses lattice params & frac_coords
        # assert canonical_crystal.matches(crystal)
        return canonical_crystal
    except: 
        return None 


def build_crystal_graph(crystal, graph_method='crystalnn'):
    try: 
        """
        """

        if graph_method == 'crystalnn':
            crystal_graph = StructureGraph.with_local_env_strategy(
                crystal, CrystalNN)
        elif graph_method == 'none':
            pass
        else:
            raise NotImplementedError

        frac_coords = crystal.frac_coords
        atom_types = crystal.atomic_numbers
        lattice_parameters = crystal.lattice.parameters
        lengths = lattice_parameters[:3]
        angles = lattice_parameters[3:]

        assert np.allclose(crystal.lattice.matrix,
                        lattice_params_to_matrix(*lengths, *angles))

        edge_indices, to_jimages = [], []
        if graph_method != 'none':
            for i, j, to_jimage in crystal_graph.graph.edges(data='to_jimage'):
                edge_indices.append([j, i])
                to_jimages.append(to_jimage)
                edge_indices.append([i, j])
                to_jimages.append(tuple(-tj for tj in to_jimage))

        atom_types = np.array(atom_types)
        lengths, angles = np.array(lengths), np.array(angles)
        edge_indices = np.array(edge_indices)
        to_jimages = np.array(to_jimages)
        num_atoms = atom_types.shape[0]

        return frac_coords, atom_types, lengths, angles, edge_indices, to_jimages, num_atoms
    except: 
        return None

In [15]:
from tqdm import tqdm

In [20]:
def graph_eval(chunk, df): 
    results = []
    #originally wrote the function to loop over all keys, but practically, I can only do 100 
    for material_id in tqdm(list(chunk.keys())[:100]):
        local_copy = build_crystal_graph(build_crystal(df[df['material_id'] == material_id]['cif'].item()))
        results.append(compare_complex_structures(local_copy, chunk[material_id]))
    return results

In [21]:
train_results = graph_eval(train_chunk, train_df)

100%|██████████| 100/100 [00:31<00:00,  3.14it/s]


In [22]:
np.mean(train_results)

1.0

In [23]:
val_results = graph_eval(val_chunk, val_df)

100%|██████████| 100/100 [00:29<00:00,  3.43it/s]


In [24]:
test_results = graph_eval(test_chunk, test_df)

100%|██████████| 100/100 [00:30<00:00,  3.31it/s]


In [25]:
np.mean(val_results)

1.0

In [26]:
np.mean(test_results)

1.0

Now we have to repeat the process that we did for the graph data before. After all the data is generated, we need to merge it back together 

In [28]:
import numpy as np
import torch 
from tqdm import tqdm

#assuming all of the data was stored as .pt dictionaries of string keys and graph values
num_workers = 100 
dataset_names =  ['train', 'test', 'val']
data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'
for name in dataset_names:
    total_dict = {}
    for worker_num in tqdm(range(num_workers)):
        chunk = torch.load(data_dir + 'graph_preloading_data/{}_{}.pt'.format(name, worker_num))
        total_dict.update(chunk)
    torch.save(total_dict, data_dir + 'graph_preloading_data/{}.pt'.format(name))
    print('Saved {}.pt'.format(name))

100%|██████████| 100/100 [00:11<00:00,  8.73it/s]


Saved train.pt


100%|██████████| 100/100 [00:04<00:00, 24.00it/s]


Saved test.pt


100%|██████████| 100/100 [00:04<00:00, 23.13it/s]


Saved val.pt


Now we just need to double check that the data was made correctly

In [53]:
import numpy as np
import torch
import pandas as pd
#load in the new train, test, and val dictionaries
data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'

train_dict = torch.load(data_dir + "graph_preloading_data/train.pt")
test_dict = torch.load(data_dir + "graph_preloading_data/test.pt")
val_dict = torch.load(data_dir + "graph_preloading_data/val.pt")

In [54]:
train_list = list(train_dict.values())
test_list = list(test_dict.values())
val_list = list(val_dict.values())

In [55]:
len(train_list)

226323

In [56]:
#check for any null values in the train, test, and val lists
print("are there nulls in the train list? ", None in train_list)
print("are there nulls in the test list? ", None in test_list)
print("are there nulls in the val list? ", None in val_list)

are there nulls in the train list?  False
are there nulls in the test list?  False
are there nulls in the val list?  False


In [39]:
#print the keys fo the null values in the train, test, and val dictionaries
null_train_keys  = [key for key, value in train_dict.items() if value is None]
print("train keys: ", null_train_keys)
null_test_keys  = [key for key, value in test_dict.items() if value is None]
print("test keys: ", null_test_keys)
null_val_keys  = [key for key, value in val_dict.items() if value is None]
print("val keys: ", null_val_keys)

train keys:  ['train_not_true_mat_id_20485', 'train_not_true_mat_id_48771', 'train_not_true_mat_id_53357', 'train_not_true_mat_id_58602', 'train_not_true_mat_id_82103', 'train_not_true_mat_id_135960', 'train_not_true_mat_id_136317', 'train_not_true_mat_id_146911', 'train_not_true_mat_id_154881', 'train_not_true_mat_id_179788']
test keys:  ['test_not_true_mat_id_34646', 'test_not_true_mat_id_50061', 'test_not_true_mat_id_59664']
val keys:  ['val_not_true_mat_id_10788', 'val_not_true_mat_id_71919']


In [40]:
#remove the null values from the train, test, and val dictionaries
for key in null_train_keys:
    del train_dict[key]

for key in null_test_keys:
    del test_dict[key]

for key in null_val_keys: 
    del val_dict[key]

In [41]:
#check to see if there are any remaining null values in the train, test, and val dictionaries
print("are there nulls in the train dict? ", None in train_dict.values())
print("are there nulls in the test dict? ", None in test_dict.values())
print("are there nulls in the val dict? ", None in val_dict.values())

are there nulls in the train dict?  False
are there nulls in the test dict?  False
are there nulls in the val dict?  False


In [43]:
#save the cleaned dictionaries to a file
torch.save(train_dict, data_dir + 'graph_preloading_data/train.pt')
torch.save(test_dict, data_dir + 'graph_preloading_data/test.pt')
torch.save(val_dict, data_dir + 'graph_preloading_data/val.pt')

So I messed up and used the train_xrd, test_xrd, val_xrd datasets instead of the train_xrd_disc_sim.csv. But should still be ok as we clean up the disc sim xrds 

In [58]:
import pandas as pd
import numpy as np
import torch 

In [59]:
train_dict = torch.load(data_dir + 'graph_preloading_data/train.pt')
test_dict = torch.load(data_dir + 'graph_preloading_data/test.pt')
val_dict = torch.load(data_dir + 'graph_preloading_data/val.pt')


In [60]:
train_keys = list(train_dict.keys())
test_keys = list(test_dict.keys())
val_keys = list(val_dict.keys())

In [73]:
data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'

#load in the data 
train_df = pd.read_csv(data_dir + 'train_xrd_disc_sim.csv')
test_df = pd.read_csv(data_dir + 'test_xrd_disc_sim.csv')
val_df = pd.read_csv(data_dir + 'val_xrd_disc_sim.csv')

In [74]:
# Manually assign names
train_df.name = 'train'
test_df.name = 'test'
val_df.name = 'val'

# Function to add material_id column
def add_material_id(df):
    df['material_id'] = df.apply(lambda row: f"{df.name}_not_true_mat_id_{row.name}", axis=1)

In [75]:
add_material_id(train_df)
add_material_id(test_df)
add_material_id(val_df)

In [76]:
train_df = train_df[train_df.material_id.isin(train_keys)]
test_df = test_df[test_df.material_id.isin(test_keys)]
val_df = val_df[val_df.material_id.isin(val_keys)]

In [77]:
#add property columns to avoid breaking the model 
train_df['formation_energy_per_atom'] = 0.0
test_df['formation_energy_per_atom'] = 0.0
val_df['formation_energy_per_atom'] = 0.0

In [78]:
print("train df and graph dict match? ", len(train_df) == len(train_dict))
print("val df and graph dict match? ", len(val_df) == len(val_dict))
print("test df and graph dict match? ", len(test_df) == len(test_dict))

train df and graph dict match?  True
val df and graph dict match?  True
test df and graph dict match?  True


In [79]:
#save the new dataframes
data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'

train_df.to_csv(data_dir + 'train_xrd_disc_sim_cleaned.csv', index=False)
test_df.to_csv(data_dir + 'test_xrd_disc_sim_cleaned.csv', index=False)
val_df.to_csv(data_dir + 'val_xrd_disc_sim_cleaned.csv', index=False)